In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from joblib import dump

In [ ]:
df = pd.read_csv('../../Dataset/H1_DataStructure.csv', parse_dates=['time'], index_col='time')
df['time'] = df.index
df['forward_return_10bar'] = (df['c'].shift(-10) - df['c']) / df['c']
df['forward_return_3bar'] = (df['c'].shift(-3) - df['c']) / df['c']
df['forward_return_5bar'] = (df['c'].shift(-5) - df['c']) / df['c']
df['forward_return_1bar'] = (df['c'].shift(-1) - df['c']) / df['c']

df['atr_norm'] = df['ATR_14'] / df['ATR_14'].rolling(252).mean()

df['vol_regime'] = pd.qcut(
    df['atr_norm'],
    q=[0, 0.33, 0.66, 1.0],
    labels=['Low', 'Medium', 'High']
)
df['position'] = None



In [2]:
class Orderblock:
    def __init__(self, data, capital=1000000):
        self.data = data
        self.position = 0
        self.trade_count = 0
        self.current_trade = 0
        self.trade_created_at = 0
        self.previous_balance = capital
        self.balance = capital
        self.capital = capital
        self.unit = 0
        self.sl_amount = 0
        self.tp_amount = 0
        self.entry = 0
        self.profit = 0
        self.loss = 0
        self.equity = []

    def analyze_gold_obs(self, displacement_mult=2.0, forward_window=10):
        """
        df: DataFrame with ['o', 'h', 'l', 'c', 'ATR_14']
        type: either bullish or bearish
        displacement_mult: How much stronger the move must be than the OB candle to count.
        forward_window: How many candles to look ahead for return after a hit.
        """
        obs = []
        active_zones = []

        for i in range(1, len(self.data) - 1):

            if self.current_trade != 1:
                if self.position == 1:
                    self.current_trade = 1
                    self.buy_position(i, amount=self.balance)

            # TP
            if self.data['h'].iloc[i] >= self.entry + (0.02 * self.entry) and self.position == 1:
                print('tp hit')
                self.sell(i, unit=self.unit)
                self.current_trade = 0
                self.position = 0
                self.tp_amount += 1
                # self.profit += 1
                self.previous_balance = self.balance
                self.equity.append({
                    'time': self.data.index[i],
                    'balance': round(self.balance)
                })

            # SL
            if self.data['l'].iloc[i] <= self.entry - (0.005 * self.entry) and self.position == 1:
                print('sl hit')
                self.sell(i, unit=self.unit)
                self.current_trade = 0
                self.position = 0
                self.sl_amount += 1
                # self.loss += 1
                self.previous_balance = self.balance
                self.equity.append({
                    'time': self.data.index[i],
                    'balance': round(self.balance)
                })

            curr = self.data.iloc[i]
            prev = self.data.iloc[i - 1]

            # --- 1. IDENTIFY NEW ORDER BLOCKS ---
            # Bullish OB: Last Bearish candle before a strong Bullish move
            if curr['c'] > curr['o'] and (curr['c'] - curr['o']) > (prev['h'] - prev['l']) * displacement_mult and \
                    prev['body'] < prev['ATR_14']:
                if prev['c'] < prev['o']:
                    active_zones.append({
                        'type': 'Bullish',
                        'top': prev['h'],
                        'bottom': prev['l'],
                        'created_at': i,
                        'created_time': self.data['time'].iloc[i],
                        'status': 'Active'
                    })

            for zone in active_zones:
                if zone['status'] != 'Active': continue

                # Check for INVALIDATION (Body Close through zone)
                if zone['type'] == 'Bullish' and curr['c'] < zone['bottom']:
                    zone['status'] = 'Invalidated'
                    continue
                hit = False
                if zone['type'] == 'Bullish':
                    # Low enters zone, but Close stays above bottom
                    if curr['l'] <= ((zone['top'] + zone['bottom']) / 2) and i - zone['created_at'] > 10:
                        hit = True
                if hit:
                    # Capture the Return
                    future_idx = min(i + forward_window, len(self.data) - 1)
                    future_price = self.data.iloc[future_idx]['c']
                    ret = ((future_price - ((zone['top'] + zone['bottom']) / 2)) / (
                                (zone['top'] + zone['bottom']) / 2)) if zone['type'] == 'Bullish' else (
                                curr['c'] - future_price)

                    obs.append({
                        'Type': zone['type'],
                        'Created_At': self.data.index[zone['created_at']],
                        'impulse_candle_size': self.data['body'].shift(-1).iloc[zone['created_at']],
                        'created_hour': self.data.index[zone['created_at']].hour,
                        'time': self.data['time'].iloc[i],
                        'distance': (self.data['time'].iloc[i] - self.data.index[
                            zone['created_at']]).total_seconds() / 3600,
                        # distance_in_hours = time_diff.total_seconds() / 3600.0
                        'vol_regime': self.data['vol_regime'].iloc[i],
                        'sessions': self.data['sessions'].iloc[i],
                        'Hit_At': self.data.index[i],
                        'l': self.data['l'].iloc[i],
                        'c': self.data['c'].iloc[i],
                        'o': self.data['o'].iloc[i],
                        'h': self.data['h'].iloc[i],
                        'hour_hit': self.data.index[i].hour,
                        # 'highest_after_hit':self.data['highest'].iloc[i],
                        # 'lowest_after_hit':self.data['lowest'].iloc[i],
                        'Return': ret,
                        'success': 1 if (ret > 0) else 0,
                        'Zone_Top': zone['top'],
                        'Zone_Bottom': zone['bottom'],
                        'zone_size': zone['top'] - zone['bottom'],
                        'atr_14': self.data['ATR_14'].iloc[i],
                    })
                    zone['status'] = 'Mitigated'  # Mark as done

            if i - self.trade_created_at > 10 and self.position == 1:
                print('going neutral')
                self.sell(i, unit=self.unit)
                self.current_trade = 0
                self.position = 0
                if self.balance > self.previous_balance:
                    self.profit += 1
                elif self.balance < self.previous_balance:
                    self.loss += 1

                self.previous_balance = self.balance
                self.equity.append({
                    'time': self.data.index[i],
                    'balance': round(self.balance)
                })

        return pd.DataFrame(obs)

    def show_data(self):
        return self.data

    def get_values(self, bar):
        date = str(self.data.index[bar])
        price = round(self.data['c'].iloc[bar], 5)
        return date, price

    def buy_position(self, bar, amount=None, unit=None):
        date, price = self.get_values(bar)
        if amount is not None:
            unit = int(amount / self.entry)
        self.balance -= unit * self.entry
        self.unit += unit
        self.trade_count += 1

        print(75 * "-")
        print(self.balance)
        print("{} | +++ Buy POSITION +++".format(date))
        print("{} |  Buying {} for {}".format(date, unit, round(price, 5)))
        perf = (self.balance - self.capital) / self.capital * 100
        # print("{} | net performance (%) = {}".format(date, round(perf, 2) ))
        print("{} | number of trades executed = {}".format(date, self.trade_count))
        print(75 * "-")

    def sell(self, bar, amount=None, unit=None):
        date, price = self.get_values(bar)
        if amount is not None:
            unit = int(amount / price)
        self.balance += unit * price
        self.unit -= unit
        self.trade_count += 1

        print(75 * "-")
        print(self.balance)
        print("{} | +++ Neutral POSITION +++".format(date))
        print("{} |  selling {} for {}".format(date, unit, round(price, 5)))
        perf = (self.balance - self.capital) / self.capital * 100
        print("{} | net performance (%) = {}".format(date, round(perf, 2)))
        print("{} | number of trades executed = {}".format(date, self.trade_count))
        print(75 * "-")


In [3]:
ob = Orderblock(df)
bullish_ob = ob.analyze_gold_obs()
session_map = {'asian': 1, 'asian/london': 2, 'london': 3, 'london/NY': 4, 'NY': 5, 'closing': 6}
bullish_ob['sessions_num'] = bullish_ob['sessions'].map(session_map)
bullish_ob['sessions_num'] = bullish_ob['sessions'].map(session_map)

regime_map = {'Low': 1, 'Medium': 2, 'High': 3}  # Adjust based on your actual labels
bullish_ob['vol_regime_num'] = bullish_ob['vol_regime'].map(regime_map)
bullish_ob['vol_regime_num'] = bullish_ob['vol_regime'].map(regime_map)

,Type,Created_At,impulse_candle_size,created_hour,time,distance,vol_regime,sessions,Hit_At,l,...,h,hour_hit,Return,success,Zone_Top,Zone_Bottom,zone_size,atr_14,sessions_num,vol_regime_num
0,Bullish,2015-01-30 15:00:00,0.742,15,2015-02-03 14:00:00,95.0,High,london/NY,2015-02-03 14:00:00,1260.780,...,1269.716,14,-0.002293,0,1264.618,1258.608,6.010,4.971643,4.0,3
1,Bullish,2015-01-29 23:00:00,0.900,23,2015-02-03 15:00:00,112.0,High,london/NY,2015-02-03 15:00:00,1256.334,...,1265.760,15,0.003054,1,1257.609,1256.533,1.076,5.338143,4.0,3
2,Bullish,2015-02-18 19:00:00,3.490,19,2015-02-20 18:00:00,47.0,High,NY,2015-02-20 18:00:00,1199.814,...,1207.777,18,0.002608,1,1201.433,1199.007,2.426,4.155929,5.0,3
3,Bullish,2015-01-02 15:00:00,4.282,15,2015-03-06 16:00:00,1513.0,High,london/NY,2015-03-06 16:00:00,1171.329,...,1176.446,16,-0.001168,0,1174.344,1169.144,5.200,4.160286,4.0,3
4,Bullish,2015-03-13 20:00:00,2.180,20,2015-03-16 11:00:00,63.0,Medium,london,2015-03-16 11:00:00,1155.246,...,1157.690,11,-0.001250,0,1156.176,1155.040,1.136,2.669000,3.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,Bullish,2025-08-05 12:00:00,7.240,12,2025-08-11 11:00:00,143.0,High,london,2025-08-11 11:00:00,3349.600,...,3358.650,11,-0.001477,0,3354.210,3349.670,4.540,10.488929,3.0,3
233,Bullish,2025-09-16 00:00:00,6.990,0,2025-09-17 01:00:00,25.0,Medium,asian,2025-09-17 01:00:00,3675.650,...,3691.900,1,-0.003347,0,3680.320,3675.840,4.480,8.033571,1.0,2
234,Bullish,2025-09-23 08:00:00,10.410,8,2025-09-23 19:00:00,11.0,High,NY,2025-09-23 19:00:00,3751.780,...,3768.030,19,0.004794,1,3757.270,3748.670,8.600,13.453571,5.0,3
235,Bullish,2025-11-18 13:00:00,0.200,13,2025-11-20 02:00:00,37.0,High,asian,2025-11-20 02:00:00,4036.400,...,4081.500,2,0.011006,1,4045.010,4029.010,16.000,23.402857,1.0,3


In [8]:
bullish_ob.to_csv('bullish_ob.csv')


In [4]:
# model creation
X = bullish_ob.drop(columns=['success', 'Type', 'time', 'Hit_At', 'Created_At', 'vol_regime', 'Return', 'sessions',
                             'vol_regime'])  # Drop non-numeric/target columns
y = bullish_ob['success']

split_idx = int(len(X) * 0.7)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 3. Train a Lightweight Random Forest
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 4. See how it did
predictions = model.predict(X_test)
print(classification_report(y_test, predictions))


              precision    recall  f1-score   support

           0       0.54      0.28      0.37        25
           1       0.69      0.87      0.77        47

    accuracy                           0.67        72
   macro avg       0.62      0.58      0.57        72
weighted avg       0.64      0.67      0.63        72



In [5]:
confusion_matrix(y_test, predictions)

array([[ 7, 18],
       [ 6, 41]])

In [7]:
dump(model, '../../models/ob_model.joblib')

['ob_model.joblib']